# Etapa 2 – Limpieza y Transformación (ETL)
Proyecto: Subtes de Buenos Aires

Objetivo: asegurar calidad de datos (nulos, duplicados, tipos, consistencia), transformar variables necesarias y dejar un dataset final listo para visualización (Etapa 3).


In [14]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None) # muestra todas las columnas
pd.set_option("display.width", 120) # ajusto el ancho


## Extract (fuentes)

Utilizo como en la etapa 1 el dataset correspondiente al año 2019 para continuar con el ETL.

In [15]:
path = "../data/raw/"

df_2019 = pd.read_csv(path + "historico_2019.csv")
df_2019.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Lima_N_Turn02,Lima,1.0,0.0,0.0,1.0
1,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Loria_N_Turn03,Loria,3.0,0.0,0.0,3.0
2,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_Q_HALL_Turn01,Plaza Miserere,3.0,0.0,0.0,3.0
3,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_S_Turn01,Plaza Miserere,6.0,0.0,0.0,6.0
4,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_S_Turn03,Plaza Miserere,10.0,0.0,0.0,10.0


In [16]:
df_2019.info() # muestro info del dataframe por si hay columnas con muchos nulos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype  
---  ------           -----  
 0   periodo          int64  
 1   fecha            object 
 2   desde            object 
 3   hasta            object 
 4   linea            object 
 5   molinete         object 
 6   estacion         object 
 7   pax_pagos        float64
 8   pax_pases_pagos  float64
 9   pax_franq        float64
 10  total            float64
dtypes: float64(4), int64(1), object(6)
memory usage: 1.0+ GB


In [17]:
df_2019.describe() # estadisticas descriptivas del dataframe

,periodo,pax_pagos,pax_pases_pagos,pax_franq,total
count,1.266234e+07,1.266234e+07,1.266234e+07,1.266234e+07,1.266234e+07
mean,2.019066e+05,2.563776e+01,1.186783e-01,1.200694e+00,2.695714e+01
std,3.444613e+00,2.832257e+01,5.024237e-01,4.085881e+00,2.939805e+01
min,2.019010e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.019040e+05,5.000000e+00,0.000000e+00,0.000000e+00,6.000000e+00
50%,2.019070e+05,1.600000e+01,0.000000e+00,0.000000e+00,1.700000e+01
75%,2.019100e+05,3.600000e+01,0.000000e+00,2.000000e+00,3.800000e+01
max,2.019120e+05,4.310000e+02,8.500000e+01,1.243800e+04,1.245900e+04


In [18]:
df_2019.isnull().sum()

periodo            0
fecha              0
desde              0
hasta              0
linea              0
molinete           0
estacion           0
pax_pagos          0
pax_pases_pagos    0
pax_franq          0
total              0
dtype: int64

## Insights
Pese a que el dataframe no tenga nulos, es notorio que posee una gran variedad de columnas con tipos de datos erroneos:

fecha    object

desde    object

hasta    object

linea    object

estacion object


In [19]:
# Conversion de fechas
df_2019['fecha'] = pd.to_datetime(df_2019['fecha'], errors='coerce')
# desde y hasta se dejan como string/object ya que representan rangos horarios
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   periodo          int64         
 1   fecha            datetime64[ns]
 2   desde            object        
 3   hasta            object        
 4   linea            object        
 5   molinete         object        
 6   estacion         object        
 7   pax_pagos        float64       
 8   pax_pases_pagos  float64       
 9   pax_franq        float64       
 10  total            float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(5)
memory usage: 1.0+ GB


### Conversión y estandarización de campos temporales

La columna `fecha` se convirtió al tipo `datetime` para asegurar una correcta interpretación temporal de los registros y habilitar análisis basados en tiempo (por día, mes o período).

Las columnas `desde` y `hasta` se mantienen como texto ya que representan rangos o franjas horarias en formato string (por ejemplo: "06:00 - 07:00").

El uso del parámetro `errors='coerce'` asegura un manejo controlado de posibles valores mal formateados en la columna `fecha`.

## Optimizacion de memoria

In [20]:
df_2019.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   periodo          int64         
 1   fecha            datetime64[ns]
 2   desde            object        
 3   hasta            object        
 4   linea            object        
 5   molinete         object        
 6   estacion         object        
 7   pax_pagos        float64       
 8   pax_pases_pagos  float64       
 9   pax_franq        float64       
 10  total            float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(5)
memory usage: 4.1 GB


In [21]:
df_2019["periodo"] = df_2019["periodo"].astype("int32") # no necesita tanto espacio de memoria como int64

for col in ["linea", "molinete", "estacion"]:
    df_2019[col] = df_2019[col].astype("category") # lo mismo aca, convierto a category que ocupa menos espacio de memoria


In [22]:
df_2019.info(memory_usage="deep") #chequeo post optimizacion de memoria


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   periodo          int32         
 1   fecha            datetime64[ns]
 2   desde            object        
 3   hasta            object        
 4   linea            category      
 5   molinete         category      
 6   estacion         category      
 7   pax_pagos        float64       
 8   pax_pases_pagos  float64       
 9   pax_franq        float64       
 10  total            float64       
dtypes: category(3), datetime64[ns](1), float64(4), int32(1), object(2)
memory usage: 1.9 GB


### Optimización de memoria
Dado el volumen del dataset (millones de filas), decidi optimizar los siguientes tipos de datos:
- `periodo` se convirtió a `int32`.
- `linea`, `molinete` y `estacion` se convirtieron a `category`, reduciendo el uso de memoria y mejorando el rendimiento en agrupaciones.


In [23]:
# Normalización de textos (consistencia semántica)
df_2019["estacion"] = (
    df_2019["estacion"].astype("string")
    .str.strip()
    .str.title()
    .astype("category")
)

df_2019["linea"] = (
    df_2019["linea"].astype("string")
    .str.strip()
    .str.upper()
    .astype("category")
)

df_2019["molinete"] = (
    df_2019["molinete"].astype("string")
    .str.strip()
    .str.upper()
    .astype("category")
)
df_2019.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LIMA_N_TURN02,Lima,1.0,0.0,0.0,1.0
1,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LORIA_N_TURN03,Loria,3.0,0.0,0.0,3.0
2,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_Q_HALL_TURN01,Plaza Miserere,3.0,0.0,0.0,3.0
3,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN01,Plaza Miserere,6.0,0.0,0.0,6.0
4,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN03,Plaza Miserere,10.0,0.0,0.0,10.0


In [24]:
df_2019.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12662343 entries, 0 to 12662342
Data columns (total 11 columns):
 #   Column           Dtype         
---  ------           -----         
 0   periodo          int32         
 1   fecha            datetime64[ns]
 2   desde            object        
 3   hasta            object        
 4   linea            category      
 5   molinete         category      
 6   estacion         category      
 7   pax_pagos        float64       
 8   pax_pases_pagos  float64       
 9   pax_franq        float64       
 10  total            float64       
dtypes: category(3), datetime64[ns](1), float64(4), int32(1), object(2)
memory usage: 772.9+ MB


In [25]:
# Cuenta de duplicados luego de la limpieza
duplicados = df_2019.duplicated().sum()
print(f"Cantidad de filas duplicadas: {duplicados}")

Cantidad de filas duplicadas: 0


In [29]:
# No hace falta porque no hay duplicados pero si hubiese usaria esto:
# df_2019 = df_2019.drop_duplicates().reset_index(drop=True) -> el reset_index es para que no me quede en la data el indice viejo como columna

df_2019.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LIMA_N_TURN02,Lima,1.0,0.0,0.0,1.0
1,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LORIA_N_TURN03,Loria,3.0,0.0,0.0,3.0
2,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_Q_HALL_TURN01,Plaza Miserere,3.0,0.0,0.0,3.0
3,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN01,Plaza Miserere,6.0,0.0,0.0,6.0
4,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN03,Plaza Miserere,10.0,0.0,0.0,10.0


In [38]:
# Variables derivadas para la Etapa 3

df_2019["hora_desde"] = pd.to_datetime(df_2019["desde"], format='%H:%M:%S').dt.hour.astype("Int8")
df_2019["hora_hasta"] = pd.to_datetime(df_2019["hasta"], format='%H:%M:%S').dt.hour.astype("Int8")
df_2019["dia_semana"] = df_2019["fecha"].dt.day_name()
df_2019["mes"] = df_2019["fecha"].dt.month.astype("Int8")

df_2019.head()

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,hora_desde,dia_semana,mes,hora_hasta
0,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LIMA_N_TURN02,Lima,1.0,0.0,0.0,1.0,8,Tuesday,1,8
1,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_LORIA_N_TURN03,Loria,3.0,0.0,0.0,3.0,8,Tuesday,1,8
2,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_Q_HALL_TURN01,Plaza Miserere,3.0,0.0,0.0,3.0,8,Tuesday,1,8
3,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN01,Plaza Miserere,6.0,0.0,0.0,6.0,8,Tuesday,1,8
4,201901,2019-01-01,08:00:00,08:15:00,LINEAA,LINEAA_MISERERE_S_TURN03,Plaza Miserere,10.0,0.0,0.0,10.0,8,Tuesday,1,8


In [41]:
#Exportacion de los datos limpios
path_clean = "../data/clean/"
df_2019.to_csv(path_clean + "historico_2019_clean.csv", index=False)

print("Datos exportados correctamente en: " + path_clean)

Datos exportados correctamente en: ../data/clean/
